# **--- GETAROUND PROJECT (pricing optimization : MODEL TRAINING AND SELECTION) ---** #

In this notebook, we will train different regression models:
1. Linear Regression (as a baseline)
2. Random Forest Regressor
3. XGBoost Regressor

Each model and its metrics will be logged into MLflow and we will evaluate the models using the following metrics:
- Mean Squared Error (MSE)
- Mean Absolute Error (MAE)
- R-squared (R2)

The model with the best performance will be assigned the "Production" alias in MLflow Model Registry. This model will then be used for making predictions in our API.

### **1. Importing Required Libraries**

In [22]:
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

### **2. Loading the Preprocessed Datasets**

In [26]:
X_train = np.load('../DATA/Data_processed/X_train_preprocessed.npy', allow_pickle=True)
X_test = np.load('../DATA/Data_processed/X_test_preprocessed.npy', allow_pickle=True)
y_train = np.load('../DATA/Data_processed/y_train.npy', allow_pickle=True)
y_test = np.load('../DATA/Data_processed/y_test.npy', allow_pickle=True)

### **3. Setting up MLflow Tracking**

In [23]:
mlflow.set_tracking_uri("http://localhost:5050")
mlflow.set_experiment("Rental_Price_Prediction")

2026/04/04 16:35:23 INFO mlflow.tracking.fluent: Experiment with name 'Rental_Price_Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location=('/Users/charlescabral/Desktop/Mounia/FORMATION JEDHA/1. '
 'JEDHA_FULLSTACK/0.PROJECTS_CERTIF/N°8_Getaround '
 'projects/N°2_Getaround_pricing/mlruns/1'), creation_time=1775313323873, experiment_id='1', last_update_time=1775313323873, lifecycle_stage='active', name='Rental_Price_Prediction', tags={}>

### **4. Training and Evaluating Models**

For each model, we will:
1. Start an MLflow run
2. Train the model
3. Evaluate the model on the test set
4. Log the model and its parameters in MLflow
5. Log the evaluation metrics in MLflow

**4.1. Linear regression (baseline)**

In [ ]:
# Linear Regression
with mlflow.start_run(run_name="Linear Regression"):
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param("model", "Linear Regression")
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    
    mlflow.sklearn.log_model(lr, "model")

2026/04/04 16:35:46 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /var/folders/bm/kyvsw80558983hqg4pthtq240000gn/T/tmp02lihopg/model/model.pkl, flavor: sklearn), fall back to return ['scikit-learn==1.3.0', 'cloudpickle==2.2.1']. Set logging level to DEBUG to see the full traceback.
/opt/anaconda3/envs/Getaround_project/lib/python3.10/site-packages/_distutils_hack/__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
/opt/anaconda3/envs/Getaround_project/lib/python3.10/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing

**4.2. Random Forest**

In [ ]:
# Random Forest Regressor  
with mlflow.start_run(run_name="Random Forest"):
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param("model", "Random Forest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    
    mlflow.sklearn.log_model(rf, "model")

**4.3. XGBoost**

In [ ]:
# XGBoost Regressor
with mlflow.start_run(run_name="XGBoost"):
    xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    xgb.fit(X_train, y_train)
    
    y_pred = xgb.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("mae", mae) 
    mlflow.log_metric("r2", r2)
    
    mlflow.sklearn.log_model(xgb, "model")

### **5. Model Selection**

Now that we have trained and logged multiple models, we can compare their performance in the MLflow UI. 

We will select the model with the lowest MSE as our best model. We will assign this model a "Production" alias in the MLflow Model Registry. This will make it easy for us to retrieve this model later for making predictions in our API.

In [25]:
# Get the best model's run_id
experiment = mlflow.get_experiment_by_name("Rental Price Prediction")
runs = mlflow.search_runs(experiment_ids=experiment.experiment_id, order_by=["metrics.mse ASC"])
best_run_id = runs.iloc[0].run_id

# Register the best model with a "Production" alias
model_uri = f"runs:/{best_run_id}/model"
mlflow.register_model(model_uri, "rental_price_predictor")
client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage(name="rental_price_predictor", 
                                      version=1, 
                                      stage="Production")

AttributeError: 'NoneType' object has no attribute 'experiment_id'

Now our best model is registered in MLflow with a "Production" alias. We can use this model for making predictions in our API by loading it using the alias.

In the API script, we can load the model like this:

```python
model = mlflow.pyfunc.load_model("models:/rental_price_predictor/Production")
```

This way, whenever we have a new best model, we can simply update the "Production" alias to point to this new model. Our API will then automatically start using the new best model.